# Experiment !!! :D

In [1]:
# IMPORTS
import time
import random
import math

from rep1.dataset import *

# """ Runtime parameters """
# assignment_nr = 1       # The assignment number. Used by the visualizer to determine what has to be visualized

In [2]:
# HELPER FUNCTIONS

def compute_centroids(cluster_points, n, d, k):
    """
    Computes the optimal set of centroids for a fixed assignment of cluster labels to the input points

    :param cluster_points: cluster points of ClusterPoint class
    :param n: number of points
    :param d: dimension
    :param k: number of clusters
    """

    cluster_sizes = [0] * k # initialize cluster sizes
    point_sum = [ClusterPoint(dimension=d) for _ in range(k)] # initialize point sums
    centroids = [CentroidPoint(dimension=d) for _ in range(k)] # initialize centroids

    for i in range(n):
        cluster_sizes[cluster_points[i].cluster_label] += 1
        point_sum[cluster_points[i].cluster_label].add(other=cluster_points[i])

    for i in range(k):
        print(f"cluster sizes {i}: {cluster_sizes[i]}")
        if cluster_sizes[i] == 0:
            continue

        point_sum[i].div(cluster_sizes[i])
        centroids[i] = point_sum[i]

    return centroids

def compute_labels(cluster_points, centroids, n, d, k):
    """
    Assigns cluster points to centroids.

    :param cluster_points: cluster pointsn of ClusterPoint class
    :param centroids: previously defined centroids
    :param n: number of points
    :param d: dimension
    :param k: number of clusters
    """

    for i in range(n):
        min_distance = 999999 # initialize high min distance
        for j in range(k):
            current_distance = cluster_points[i].sq_distance_to(other=centroids[j])
            if current_distance < min_distance:
                min_distance = current_distance
                cluster_label = j
        cluster_points[i].cluster_label = cluster_label

    return cluster_points

def read(path_in):
    """
    Reads the input set, clusters the points and writes to output

    :param path_in:     location of the input set
    :param path_out:    location to print the output
    """
    # read input from file
    try:
        with open(path_in, "r") as f:
            input_obj = Dataset.read_input(f)
            return input_obj
    except IOError:
        print("Could not read input file: " + path_in, file=sys.stderr)
        return None

def uniform_random() -> float:
    """
    Find x uniformly at random such that 0 < x < 1.
    """
    none = True

    while none:
        x = random.random()

        if 0 < x < 1:
            return x

In [4]:
# INITIALIZE CENTROIDS

def initialize_centroids(input_obj, method="first_k"):
    """
    Calculates the initial centroid placement

    :param input_obj:   the input object
    :param method: initialization method, possible options are first_k, gonzales, kmeans++
    :return:            a list of k centroids in the plane
    """
    if method=="first_k":
        ### IMPLEMENTATION: FIRST K CLUSTER POINTS AS INITIAL CENTROIDS
        centroids = [CentroidPoint(p.dimension, list(p.coords)) for p in input_obj.cluster_points[:input_obj.k]]

    if method=="gonzales":
        ### IMPLEMENTATION: GONZALEZ
        cluster_points = input_obj.cluster_points
        centroids = [CentroidPoint(dimension=input_obj.d) for _ in range(input_obj.k)] # initialize centroids # TODO not needed
        min_distances = [0] * input_obj.n

        # initialize first centroids with first input point
        centroids[0] = cluster_points[0]

        for j in range(input_obj.n):
            min_distances[j] = 999999 # initialize high min distances

        for i in range(1, input_obj.k):
            # update distance to closest centroid for each input point
            for j in range(input_obj.n):
                x = cluster_points[j].sq_distance_to(other=centroids[i-1])
                if min_distances[j] > x:
                    min_distances[j] = x

            # compute input point which is farthest from current set of centroids
            max_distance = 0 # initialize lowest max distance

            for j in range(input_obj.n):
                if max_distance < min_distances[j]:
                    max_distance = min_distances[j]
                    farthest_point = cluster_points[j]

            # add farthest point to set of centroids
            centroids[i] = farthest_point

    if method=="kmeans++":
        ### IMPLEMENTATION: KMEANS++
        points, n, d, k = input_obj.cluster_points, input_obj.n, input_obj.d, input_obj.k
        min_distances = [0] * n
        cumulative = [0] * n

        centroids = [None] * k # initialise before indexing

        x = uniform_random()
        sampled_index = math.ceil(x * n)
        centroids[0] = points[sampled_index]

        for j in range(n):
            min_distances[j] = float('inf')

        for i in range(1, k):
            for j in range(n):
                x = points[j].sq_distance_to(centroids[i-1])

                if min_distances[j] > x:
                    min_distances[j] = x

            cumulative[0] = min_distances[0] * min_distances[0]
            for j in range(1, n):
                cumulative[j] = cumulative[j-1] + (min_distances[j] * min_distances[j])

            x = uniform_random()
            x = x * cumulative[n-1]

            if x <= cumulative[0]:
                sampled_index = 0
            else:
                for j in range(1, n):
                    if (x > cumulative[j-1]) and (x <= cumulative[j]):
                        sampled_index = j

            centroids[i] = points[sampled_index]

    if method == 'random':
        ### IMPLEMENTATION: RANDOM K
        centroids = [CentroidPoint(p.dimension, list(p.coords)) for p in random.sample(input_obj.cluster_points, input_obj.k)]

    return centroids

In [5]:
# PERFORM CLUSTERING

def cluster(input_obj, method="first_k"):
    """
    Perform k-means clustering on the input set

    :param input_obj:   the input object
    :return:            a list of k centroids in the plane
    """
    # initalize centroids
    centroids_old = initialize_centroids(input_obj, method=method)

    # obtain n, d, k once
    # initialize cluster points
    n = input_obj.n
    d = input_obj.d
    k = input_obj.k
    cluster_points = input_obj.cluster_points

    # repeat until centroids did not change in the last iteration
    i = 0
    while True:
        # print('iteration', i)

        cluster_points = compute_labels(cluster_points, centroids_old, n, d, k)
        centroids_new = compute_centroids(cluster_points, n, d, k)

        # stop if unchanged
        if centroids_new == centroids_old:
            break

        centroids_old = centroids_new
        i+=1

    centroids = centroids_new

    return centroids, i

In [6]:
# RUN AND WRITE TO OUTPUT

def run(path_in, path_out, method):
    """
    Reads the input dataset, runs k-means, writes output, and returns metrics.

    :param path_in:   location of the input file
    :param path_out:  location to write the output file
    :param method:    initialisation method ('gonzales' or 'kmeans++')
    :return:          (sse, n_iterations, wall_clock_seconds)
                        sse              – final sum of squared errors (inertia)
                        n_iterations     – Lloyd iterations until convergence
                        wall_clock_secs  – total wall-clock runtime in seconds
    """
    # ── Read input ────────────────────────────────────────────────────────────
    try:
        with open(path_in, "r") as f:
            input_obj = Dataset.read_input(f)
    except IOError:
        print("Could not read input file: " + path_in, file=sys.stderr)
        return None, None, None

    # ── Run clustering (timed) ────────────────────────────────────────────────
    t_start = time.perf_counter()                      # high-resolution timer
    centroids, n_iterations = cluster(input_obj, method=method)
    t_end   = time.perf_counter()

    wall_clock_secs = t_end - t_start

    # ── Compute metrics ───────────────────────────────────────────────────────
    assert len(centroids) == input_obj.k

    sse = input_obj.score(centroids)                   # sum of squared errors

    # ── Write output file ─────────────────────────────────────────────────────
    try:
        input_obj.write_output(centroids, path_out, 1)
    except IOError:
        print("Could not write output to file: " + path_out, file=sys.stderr)

    # ── Diagnostic stderr ─────────────────────────────────────────────────────
    print(f"[{method}] k={input_obj.k}  "
          f"SSE={sse:.3f}  "
          f"iters={n_iterations}  "
          f"time={wall_clock_secs:.3f}s",
          file=sys.stderr)

    return sse, n_iterations, wall_clock_secs

In [7]:
%%time
# RUN EXPERIMENT

# Runs gonzales and kmeans++ for every (k, r) pair.
# Writes per-run output files to output/ and a combined summary to results/.

import os
import sys
from collections import defaultdict

# Experiment parameters
k_values = [2, 5, 10, 20, 50, 100]
R        = 30                                 # runs per k (seeds 1, ..., 10)
methods  = ["gonzales", "kmeans++", "random"]

INPUT_DIR   = "input"
OUTPUT_DIR  = "output"
RESULTS_DIR = "results"
os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Storage: results[(method, k)] = list of (sse, iters, time)
results = defaultdict(list)

total_runs = len(methods) * len(k_values) * R
run_nr     = 0

for method in methods:
    for k in k_values:
        for r in range(1, R + 1):
            run_nr += 1
            path_in  = os.path.join(INPUT_DIR,  f"k{k}_R{r}.in")
            path_out = os.path.join(OUTPUT_DIR, f"{method}_k{k}_R{r}.out")

            print(f"[{run_nr}/{total_runs}] method={method}  k={k}  R={r}",
                  file=sys.stderr)

            sse, iters, t = run(path_in, path_out, method)

            if sse is not None:            # guard against read errors
                results[(method, k)].append((sse, iters, t))

# Compute averages and write summary
summary_path = os.path.join(RESULTS_DIR, "summary.txt")

col_w = 14   # column width for alignment

header = (
    f"{'Method':<12} "
    f"{'k':>{col_w}} "
    f"{'SSE_mean':>{col_w}} "
    f"{'SSE_std':>{col_w}} "
    f"{'Iters_mean':>{col_w}} "
    f"{'Iters_std':>{col_w}} "
    f"{'Time_mean(s)':>{col_w}} "
    f"{'Time_std(s)':>{col_w}}"
)
separator = "-" * len(header)

lines = [
    "K-Means Initialization Experiment — Summary",
    f"Runs per (method, k): {R}  |  n=10,000  |  d=2  |  cluster_std=0.3",
    separator,
    header,
    separator,
]

for method in methods:
    for k in k_values:
        runs = results[(method, k)]
        if not runs:
            continue

        sses, iters_list, times = zip(*runs)

        sse_mean   = sum(sses)        / len(sses)
        sse_std    = (sum((x - sse_mean)**2   for x in sses)        / len(sses)) ** 0.5
        iter_mean  = sum(iters_list)  / len(iters_list)
        iter_std   = (sum((x - iter_mean)**2  for x in iters_list)  / len(iters_list)) ** 0.5
        time_mean  = sum(times)       / len(times)
        time_std   = (sum((x - time_mean)**2  for x in times)       / len(times)) ** 0.5

        lines.append(
            f"{method:<12} "
            f"{k:{col_w}d} "
            f"{sse_mean:{col_w}.2f} "
            f"{sse_std:{col_w}.2f} "
            f"{iter_mean:{col_w}.2f} "
            f"{iter_std:{col_w}.2f} "
            f"{time_mean:{col_w}.4f} "
            f"{time_std:{col_w}.4f}"
        )
    lines.append("")   # blank line between methods

lines.append(separator)

with open(summary_path, "w") as f:
    f.write("\n".join(lines) + "\n")

print(f"\nSummary written to {summary_path}")

# Also print to notebook
print("\n".join(lines))

[1/180] method=random  k=2  R=1
[random] k=2  SSE=1799.906  iters=1  time=0.040s
[2/180] method=random  k=2  R=2
[random] k=2  SSE=1813.352  iters=2  time=0.056s
[3/180] method=random  k=2  R=3


cluster sizes 0: 5000
cluster sizes 1: 5000
cluster sizes 0: 5000
cluster sizes 1: 5000
cluster sizes 0: 3008
cluster sizes 1: 6992
cluster sizes 0: 5000
cluster sizes 1: 5000
cluster sizes 0: 5000
cluster sizes 1: 5000
cluster sizes 0: 2047
cluster sizes 1: 7953
cluster sizes 0: 5000
cluster sizes 1: 5000
cluster sizes 0: 5000
cluster sizes 1: 5000
cluster sizes 0: 7685
cluster sizes 1: 2315
cluster sizes 0: 5002
cluster sizes 1: 4998
cluster sizes 0: 5000
cluster sizes 1: 5000
cluster sizes 0: 5000
cluster sizes 1: 5000
cluster sizes 0: 5382
cluster sizes 1: 4618
cluster sizes 0: 5152
cluster sizes 1: 4848


[random] k=2  SSE=1782.158  iters=2  time=0.066s
[4/180] method=random  k=2  R=4
[random] k=2  SSE=1780.564  iters=3  time=0.076s
[5/180] method=random  k=2  R=5


cluster sizes 0: 5060
cluster sizes 1: 4940
cluster sizes 0: 5023
cluster sizes 1: 4977
cluster sizes 0: 5013
cluster sizes 1: 4987
cluster sizes 0: 5012
cluster sizes 1: 4988
cluster sizes 0: 5012
cluster sizes 1: 4988
cluster sizes 0: 6841
cluster sizes 1: 3159
cluster sizes 0: 5000
cluster sizes 1: 5000
cluster sizes 0: 5000
cluster sizes 1: 5000


[random] k=2  SSE=1689.040  iters=6  time=0.144s
[6/180] method=random  k=2  R=6
[random] k=2  SSE=1775.550  iters=2  time=0.071s
[7/180] method=random  k=2  R=7


...

[random] k=100  SSE=1897.839  iters=35  time=27.482s
[179/180] method=random  k=100  R=29


cluster sizes 0: 90
cluster sizes 1: 175
cluster sizes 2: 40
cluster sizes 3: 132
cluster sizes 4: 104
cluster sizes 5: 170
cluster sizes 6: 181
cluster sizes 7: 103
cluster sizes 8: 141
cluster sizes 9: 146
cluster sizes 10: 59
cluster sizes 11: 133
cluster sizes 12: 97
cluster sizes 13: 41
cluster sizes 14: 328
cluster sizes 15: 353
cluster sizes 16: 38
cluster sizes 17: 97
cluster sizes 18: 12
cluster sizes 19: 85
cluster sizes 20: 71
cluster sizes 21: 93
cluster sizes 22: 68
cluster sizes 23: 26
cluster sizes 24: 88
cluster sizes 25: 17
cluster sizes 26: 34
cluster sizes 27: 140
cluster sizes 28: 38
cluster sizes 29: 38
cluster sizes 30: 61
cluster sizes 31: 130
cluster sizes 32: 153
cluster sizes 33: 101
cluster sizes 34: 41
cluster sizes 35: 72
cluster sizes 36: 120
cluster sizes 37: 30
cluster sizes 38: 94
cluster sizes 39: 127
cluster sizes 40: 105
cluster sizes 41: 126
cluster sizes 42: 230
cluster sizes 43: 72
cluster sizes 44: 135
cluster sizes 45: 29
cluster sizes 46: 400
c

[random] k=100  SSE=3012.044  iters=24  time=18.802s
[180/180] method=random  k=100  R=30


cluster sizes 0: 79
cluster sizes 1: 42
cluster sizes 2: 119
cluster sizes 3: 97
cluster sizes 4: 97
cluster sizes 5: 106
cluster sizes 6: 111
cluster sizes 7: 82
cluster sizes 8: 80
cluster sizes 9: 41
cluster sizes 10: 103
cluster sizes 11: 182
cluster sizes 12: 150
cluster sizes 13: 193
cluster sizes 14: 91
cluster sizes 15: 45
cluster sizes 16: 52
cluster sizes 17: 88
cluster sizes 18: 83
cluster sizes 19: 107
cluster sizes 20: 104
cluster sizes 21: 308
cluster sizes 22: 25
cluster sizes 23: 99
cluster sizes 24: 270
cluster sizes 25: 44
cluster sizes 26: 133
cluster sizes 27: 78
cluster sizes 28: 38
cluster sizes 29: 175
cluster sizes 30: 101
cluster sizes 31: 82
cluster sizes 32: 237
cluster sizes 33: 69
cluster sizes 34: 100
cluster sizes 35: 24
cluster sizes 36: 25
cluster sizes 37: 58
cluster sizes 38: 70
cluster sizes 39: 100
cluster sizes 40: 41
cluster sizes 41: 54
cluster sizes 42: 227
cluster sizes 43: 94
cluster sizes 44: 21
cluster sizes 45: 139
cluster sizes 46: 190
clu

[random] k=100  SSE=2088.413  iters=33  time=26.332s


In [8]:
run("input/k2_R1.in", "rep1/output/test.out", "kmeans++")

cluster sizes 0: 5000
cluster sizes 1: 5000
cluster sizes 0: 5000
cluster sizes 1: 5000


[kmeans++] k=2  SSE=1799.906  iters=1  time=0.048s


(1799.9062965507676, 1, 0.047841899999184534)